In [1]:
import os
import sys
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
import coiled
from dask.distributed import Client, LocalCluster
import dask
import zarr
import gcsfs
import json
import yaml
import re
import logging
from IPython.core.magic import register_cell_magic
@register_cell_magic
def comment(line, cell):
    # This magic does nothing, effectively ignoring the cell's content
    pass
# Add app to path
sys.path.insert(0, "/home/stefan/CRS/CRS.ZarrPipelines/app")
from utils import HazardConfig, get_thresholds, score
from domain import scoring

dask.config.set({"array.rechunk.method": "p2p"})
print(os.getcwd())

/home/stefan/CRS/CRS.ZarrPipelines/app/scripts


/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


## Data Paths

In [2]:
INPUT_PATH = 'gs://crs_climate_data_public/PHYSICAL_HAZARDS_WORKING_FOLDER/LS/zarrs/processed_unscored/LS.zarr/'
OUTPUT_PATH_1_to_5 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_5/LS.zarr'
OUTPUT_PATH_1_to_3 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_3/LS.zarr'
OUTPUT_PATH_1_to_10 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_10/LS.zarr'
OUTPUT_PATH_1_to_100_norm = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_100_norm/LS.zarr'

# Spin up Coiled Cluster

In [3]:
%%comment
# Initialize cluster
cluster = coiled.Cluster(
    n_workers=15,
    scheduler_vm_types="e2-standard-8",
    worker_vm_types="e2-standard-4",
    region="europe-west8",
    name="wonder",
    shutdown_on_close=False,
    idle_timeout="2 hours"
)
client = Client(cluster)
cluster.adapt(minimum=15, maximum=20)
logging.info(f"Cluster setup complete: {client}")

## Hazard Data

In [4]:
ds = xr.open_zarr(INPUT_PATH)
ds

<xarray.Dataset> Size: 56GB
Dimensions:         (time: 4, scenario: 2, model: 1, metric: 1, statistic: 5,
                     lat: 15841, lon: 43200)
Coordinates:
  * lat             (lat) float64 127kB 72.0 71.99 71.98 ... -59.99 -60.0 -60.0
  * lon             (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0
  * metric          (metric) <U28 112B 'mean_days_per_year_above_p95'
  * model           (model) <U8 32B 'ENSEMBLE'
  * scenario        (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic       (statistic) <U6 120B 'mean' 'max' 'min' 'std' 'median'
  * time            (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
Data variables:
    ari             (time, scenario, model, metric, statistic, lat, lon) float16 55GB dask.array<chunksize=(1, 1, 1, 1, 1, 512, 512), meta=np.ndarray>
    susceptibility  (lat, lon) float16 1GB dask.array<chunksize=(512, 512), meta=np.ndarray>

# GADM data

# Step 1 : Create Unscored data according to config instructions

In [5]:
# Initialize config
config = HazardConfig()

Found config at: /home/stefan/CRS/CRS.ZarrPipelines/app/config/config.yaml


In [6]:
# Get all thresholds for LS
ls_all = config.get_all_thresholds('LS')
print("All LS thresholds:")
print(ls_all)

All LS thresholds:
{'hazard': 'Landslide', 'code': 'LS', 'metrics': {'landslide_proxy': {'thresholds_ari.5_point': {'thresholds': [0, 5, 10, 15, 20, inf], 'scores': [1, 2, 3, 4, 5, 6]}, 'thresholds_ari.10_point': {'thresholds': [0, 2.5, 5, 7.5, 10, 12.5, 15, 17.5, 20, 22.5, inf], 'scores': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}, 'thresholds_susceptibility.5_point': {'thresholds': [1, 2, 3, 4, 5], 'scores': [1, 2, 3, 4, 5]}, 'thresholds_susceptibility.10_point': {'thresholds': [1, 2, 3, 4, 5], 'scores': [2, 4, 6, 8, 10]}, 'thresholds_final.5_point': {'thresholds': [0, 5, 10, 15, 20, 25], 'scores': [1, 2, 3, 4, 5]}, 'thresholds_final.10_point': {'thresholds': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100], 'scores': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}}}}


# 1-5

In [7]:
thresh_ari_5, scores_ari_5 = config.get_thresholds('LS', '5', threshold_type='thresholds_ari')
thresh_ari_5

[0, 5, 10, 15, 20, inf]

In [8]:
thresh_susc_5, scores_susc_5 = config.get_thresholds('LS', '5', threshold_type='thresholds_susceptibility')
thresh_susc_5

[1, 2, 3, 4, 5]

In [9]:
thresh_final_5, scores_final_5 = config.get_thresholds('LS', '5', threshold_type='thresholds_final')
thresh_final_5

[0, 5, 10, 15, 20, 25]

# 1-10

In [10]:
thresh_ari_10, c = config.get_thresholds('LS', '10', threshold_type='thresholds_ari')
thresh_ari_10

[0, 2.5, 5, 7.5, 10, 12.5, 15, 17.5, 20, 22.5, inf]

In [11]:
thresh_ari_10, scores_susc_10 = config.get_thresholds('LS', '10', threshold_type='thresholds_susceptibility')
thresh_ari_10

[1, 2, 3, 4, 5]

In [12]:
thresh_final_10, scores_final_10 = config.get_thresholds('LS', '10', threshold_type='thresholds_final')
thresh_final_10

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# Step 2: Score Data according to bins with digitize

## 1-5

### 2.1 Score ARI

In [13]:
scored_ari_5 = scoring.score_zarr(
    ds['ari'],
    thresh_ari_5,
    scores_susc_5,
    metric = "ari_1_to_5"
)
scored_ari_5

Scoring array with shape (4, 2, 1, 1, 5, 15841, 43200)
Thresholds: [0, 5, 10, 15, 20, inf]
Scores: [1, 2, 3, 4, 5]
Setting metric coordinate to: ari_1_to_5


<xarray.Dataset> Size: 219GB
Dimensions:    (lat: 15841, lon: 43200, model: 1, scenario: 2, statistic: 5,
                time: 4)
Coordinates:
  * lat        (lat) float64 127kB 72.0 71.99 71.98 71.97 ... -59.99 -60.0 -60.0
  * lon        (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * model      (model) <U8 32B 'ENSEMBLE'
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U6 120B 'mean' 'max' 'min' 'std' 'median'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric     <U10 40B 'ari_1_to_5'
Data variables:
    score      (time, scenario, model, statistic, lat, lon) float64 219GB dask.array<chunksize=(1, 1, 1, 1, 512, 512), meta=np.ndarray>

In [14]:
scored_susc_5 = scoring.score_zarr(
    ds['susceptibility'],
    thresh_ari_5,
    scores_susc_5
)
scored_susc_5

Scoring array with shape (15841, 43200)
Thresholds: [0, 5, 10, 15, 20, inf]
Scores: [1, 2, 3, 4, 5]


<xarray.Dataset> Size: 5GB
Dimensions:  (lat: 15841, lon: 43200)
Coordinates:
  * lat      (lat) float64 127kB 72.0 71.99 71.98 71.97 ... -59.99 -60.0 -60.0
  * lon      (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Data variables:
    score    (lat, lon) float64 5GB dask.array<chunksize=(512, 512), meta=np.ndarray>

In [15]:
final = scored_ari_5 * scored_susc_5
final

<xarray.Dataset> Size: 219GB
Dimensions:    (lat: 15841, lon: 43200, model: 1, scenario: 2, statistic: 5,
                time: 4)
Coordinates:
  * lat        (lat) float64 127kB 72.0 71.99 71.98 71.97 ... -59.99 -60.0 -60.0
  * lon        (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * model      (model) <U8 32B 'ENSEMBLE'
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U6 120B 'mean' 'max' 'min' 'std' 'median'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric     <U10 40B 'ari_1_to_5'
Data variables:
    score      (time, scenario, model, statistic, lat, lon) float64 219GB dask.array<chunksize=(1, 1, 1, 1, 512, 512), meta=np.ndarray>

In [17]:
scored_final_5 = scoring.score_zarr(
    final['score'],
    thresh_ari_5,
    scores_susc_5,
    metric = 'landslide_proxy'
)
scored_final_5

Scoring array with shape (4, 2, 1, 5, 15841, 43200)
Thresholds: [0, 5, 10, 15, 20, inf]
Scores: [1, 2, 3, 4, 5]
Setting metric coordinate to: landslide_proxy


<xarray.Dataset> Size: 219GB
Dimensions:    (lat: 15841, lon: 43200, model: 1, scenario: 2, statistic: 5,
                time: 4)
Coordinates:
  * lat        (lat) float64 127kB 72.0 71.99 71.98 71.97 ... -59.99 -60.0 -60.0
  * lon        (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * model      (model) <U8 32B 'ENSEMBLE'
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U6 120B 'mean' 'max' 'min' 'std' 'median'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric     <U15 60B 'landslide_proxy'
Data variables:
    score      (time, scenario, model, statistic, lat, lon) float64 219GB dask.array<chunksize=(1, 1, 1, 1, 512, 512), meta=np.ndarray>

# Step 3: GADM Aggregations
## https://earthmover.io/blog/vector-datacube-pt1/